In [ ]:
import os
from dotenv import load_dotenv

# טעינת המפתחות מקובץ ה-.env שלך
load_dotenv()

COHERE_API_KEY = os.getenv("COHERE_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

# בדיקה שהכל נטען (אל תדאגי, זה לא ידפיס את המפתח המלא)
print(f"Cohere Key loaded: {bool(COHERE_API_KEY)}")
print(f"Pinecone Key loaded: {bool(PINECONE_API_KEY)}")

In [ ]:
from llama_index.core import SimpleDirectoryReader

# הגדירי את הנתיב לפרויקט הדאטה
data_path = "C:/Users/user1/Desktop/תכנותתת/שנה ב/AI/RAG/data-project"

# טעינת כל התיקיות (כולל הנסתרות שמתחילות בנקודה)
reader = SimpleDirectoryReader(
    input_dir=data_path,
    recursive=True,
    required_exts=[".md"],
    exclude_hidden=False
)

documents = reader.load_data()

print(f"נטענו {len(documents)} מסמכים.")

In [ ]:
from llama_index.core.node_parser import SentenceSplitter

# הגדרת החותך: כל חתיכה תהיה בערך 512 תווים עם חפיפה קטנה
parser = SentenceSplitter(chunk_size=512, chunk_overlap=20)
nodes = parser.get_nodes_from_documents(documents)

print(f"המסמכים פוצלו ל-{len(nodes)} חתיכות (Nodes).")

In [ ]:
from llama_index.embeddings.cohere import CohereEmbedding
from llama_index.core import Settings

# הגדרת מודל ה-Embedding
embed_model = CohereEmbedding(
    api_key=COHERE_API_KEY,
    model_name="embed-multilingual-v3.0", # מודל מעולה שתומך בעברית
)

# נגדיר ל-LlamaIndex להשתמש במודל הזה כברירת מחדל
Settings.embed_model = embed_model

print("מודל ה-Embedding הוגדר בהצלחה!")

In [ ]:
from pinecone import Pinecone
from llama_index.vector_stores.pinecone import PineconeVectorStore
from llama_index.core import VectorStoreIndex, StorageContext
import os

# 1. התחברות ל-Pinecone עם ביטול אימות SSL (בגלל נטפרי)
pc = Pinecone(
    api_key=PINECONE_API_KEY,
    ssl_verify=False  # התיקון שעבד לך
)

# 2. גישה לאינדקס שיצרת באתר (ודאי שהשם תואם למה שמופיע ב-Dashboard)
pinecone_index = pc.Index("rag-lesson")

# 3. הגדרת ה-Vector Store בתוך LlamaIndex
vector_store = PineconeVectorStore(pinecone_index=pinecone_index,namespace="")

# 4. יצירת הקשר אחסון (Storage Context)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# 5. יצירת האינדקס מתוך ה-Nodes שחתכנו קודם
# זה השלב שבו המידע נשלח ל-Cohere לתרגום ול-Pinecone לשמירה
index = VectorStoreIndex(
    nodes, 
    storage_context=storage_context
)

print("האינדקס נוצר בהצלחה! הנתונים נמצאים עכשיו ב-Pinecone.")

In [ ]:
from llama_index.llms.gemini import Gemini
from llama_index.core import Settings, PromptTemplate, get_response_synthesizer
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.postprocessor import SimilarityPostprocessor
import os

# 1. הגדרת Gemini כמודל השפה (LLM)
# הוא יהיה אחראי על ניסוח התשובה הסופית
Settings.llm = Gemini(
    api_key=GOOGLE_API_KEY, 
    model_name="models/gemini-2.5-flash", 
    temperature=0.1 ,
    transport="rest"
)
# 2. ניסוח הפרומפט המושלם - שיענה בשפת השאלה ורק לפי המידע שסופק
qa_prompt_tmpl_str = (
    "Context information is below.\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "You are a professional technical assistant. Based ONLY on the context provided above (and not your general knowledge), "
    "answer the following query. \n"
    "CRITICAL RULES:\n"
    "1. If the answer is not in the context, say: 'מצטער, המידע הזה לא מופיע בתיעוד הפרויקט'.\n"
    "2. Always respond in the SAME LANGUAGE as the user's question.\n"
    "3. If there are contradictions between tools (e.g., Windsurf vs Copilot), highlight them.\n"
    "Query: {query_str}\n"
    "Answer: "
)
qa_prompt_tmpl = PromptTemplate(qa_prompt_tmpl_str)

# 3. הגדרת רכיבי השליפה והסינתזה
retriever = VectorIndexRetriever(index=index, similarity_top_k=3)
node_postprocessor = SimilarityPostprocessor(similarity_cutoff=0.0)

response_synthesizer = get_response_synthesizer(
    response_mode="tree_summarize",
    text_qa_template=qa_prompt_tmpl
)

# 4. יצירת המנוע הסופי
custom_query_engine = RetrieverQueryEngine(
    retriever=retriever,
    response_synthesizer=response_synthesizer,
    node_postprocessors=[node_postprocessor],
)

print("מנוע Gemini הוגדר בהצלחה עם הפרומפט המותאם!")

In [ ]:
import gradio as gr

# עיצוב Dark-Tech יוקרתי ומקצועי
tech_css = """
/* רקע כללי כהה והייטקי */
.gradio-container { 
    direction: rtl; 
    background-color: #0b0f19 !important; 
    font-family: 'Inter', 'Segoe UI', sans-serif !important;
}

/* כותרת מעוצבת */
#header-text { 
    text-align: center; 
    color: #e2e8f0; 
    padding: 20px;
    border-bottom: 1px solid #1e293b;
}

/* עיצוב בועות הצ'אט */
.message { 
    border-radius: 8px !important; 
    font-size: 0.95rem !important;
    line-height: 1.6 !important;
}

/* בועת המשתמש - כחול הייטק עדין */
.user { 
    background-color: #1e293b !important; 
    color: #f1f5f9 !important; 
    border: 1px solid #334155 !important;
}

/* בועת הבוט - אפור כהה מקצועי */
.bot { 
    background-color: #0f172a !important; 
    color: #cbd5e1 !important; 
    border: 1px solid #1e293b !important;
}

/* עיצוב שדה הטקסט */
#query-input textarea {
    background-color: #1e293b !important;
    color: #f1f5f9 !important;
    border: 1px solid #334155 !important;
    border-radius: 4px !important;
}

/* כפתורי שליטה - מראה שטוח ונקי */
.lg.primary {
    background: linear-gradient(90deg, #3b82f6 0%, #2563eb 100%) !important;
    border: none !important;
}

footer { display: none !important; }
"""

def chat_response(message, history):
    # הפעלת המנוע
    response = custom_query_engine.query(message)
    return str(response)
    
    

# בניית הממשק במבנה Blocks
# בניית הממשק במבנה Blocks הכי מעודכן ל-2026
with gr.Blocks(css=tech_css, theme=gr.themes.Default()) as demo:
    with gr.Row(elem_id="header-text"):
        gr.Markdown("""
        # 💠 PROJECT KNOWLEDGE GRAPH
        **Advanced RAG Interface | Gemini 2.5 Pro**
        """)
    
    with gr.Column():
        # יצירת הצ'אטבוט
        chatbot = gr.Chatbot(
            show_label=False, 
            height=550
        )
        
        # יצירת ממשק הצ'אט עם השמות המעודכנים של הכפתורים
        chat_interface = gr.ChatInterface(
            fn=chat_response,
            chatbot=chatbot,
            textbox=gr.Textbox(
                placeholder="הזן שאילתה טכנית לניתוח הנתונים...", 
                elem_id="query-input",
                container=False, 
                scale=7
            ),
            # בגרסה 5 משתמשים בשמות האלו:
            submit_btn="RUN QUERY ⚡",
            stop_btn="STOP 🛑",
            # אם רוצים לבטל כפתורים, פשוט לא מגדירים אותם או מגדירים כ-None
        )

# הרצה
if __name__ == "__main__":
    demo.launch(debug=True)